# Build split + gold JSONL, run baselines

This notebook covers two pipeline steps:

1. **Build the stratified split** — assigns languages to LRL/MRL/HRL, samples 100
   `(language, feature)` pairs per `(resource_group × feature_type)` cell for test
   and dev, saves `split.json` and `gold_eval.jsonl`.

2. **Run baselines** — fits Random, Mean, kNN, and SoftImpute on the implicit train
   set (all observed pairs not in test/dev), scores against the test gold items,
   and saves a report JSON.

Run from the repo root: `jupyter lab notebooks/build_split_and_baselines.ipynb`

In [11]:
import json
from pathlib import Path
from lkb.kb.uriel_plus import URIELPlus
from lkb.eval.splits import Split
from lkb.eval.evaluator import Evaluator

In [5]:
kb = URIELPlus.from_artifacts(
    ".",
    typology_path="../data/derived/URIELPlus_Union.csv",
)

print(f"Languages: {len(kb.languages):,}")
print(f"Features:  {len(kb.features):,}")

obs = kb.observed_mask()
print(f"Observed entries: {obs.sum():,}  ({obs.mean()*100:.1f}% of matrix)")

Languages: 8,171
Features:  800
Observed entries: 888,741  (13.6% of matrix)


In [9]:
split = Split.stratified(
    kb,
    n_per_cell=100,
    lrl_frac=0.33,
    mrl_frac=0.34,
    min_observed=1,
    include_dev=True,
    seed=42,
)

split.save("../data/splits/splits_v2.json")
print(f"Test pairs: {len(split.test):,}")
print(f"Dev  pairs: {len(split.dev):,}")

print()
print("Cell summary (from meta):")
for cell, stats in split.meta["cells"].items():
    print(f"  {cell:<20}  available={stats['available']:>6,}  test={stats['test']}  dev={stats['dev']}")

Test pairs: 1,200
Dev  pairs: 1,200

Cell summary (from meta):
  hrl/INV               available=209,230  test=100  dev=100
  hrl/M                 available=80,051  test=100  dev=100
  hrl/P                 available=29,960  test=100  dev=100
  hrl/S                 available=191,264  test=100  dev=100
  lrl/INV               available= 6,268  test=100  dev=100
  lrl/M                 available=27,192  test=100  dev=100
  lrl/P                 available= 2,584  test=100  dev=100
  lrl/S                 available=62,612  test=100  dev=100
  mrl/INV               available=117,110  test=100  dev=100
  mrl/M                 available=50,845  test=100  dev=100
  mrl/P                 available=16,603  test=100  dev=100
  mrl/S                 available=95,022  test=100  dev=100


In [13]:
gold_items = Evaluator.gold_items_from_split(kb, split, partition="test")

GOLD_OUT = Path("../data/benchmark/gold_eval_2.jsonl")
GOLD_OUT.parent.mkdir(parents=True, exist_ok=True)
with GOLD_OUT.open("w", encoding="utf-8") as f:
    for item in gold_items:
        record = {
            "id": item.id,
            "resource_group": item.resource_group,
            "language": item.language,
            "feature": item.feature,
            "gold_value": item.gold_value,
            "allowed_values": list(item.allowed_values),
        }
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"{len(gold_items)} items")

1200 items


## 4  Run baselines

Each baseline is fitted on **all observed pairs not in test or dev** (the implicit train set), then scored against the test gold items.

In [ ]:
import numpy as np
from lkb.impute.baselines import RandomImputer, MeanImputer, KNNImputer
from lkb.impute.softimpute import SoftImputeImputer

# Pairs the baselines need to predict
test_pairs = [(item.language, item.feature) for item in gold_items]

BASELINES = [
    RandomImputer(seed=SEED),
    MeanImputer(),
    KNNImputer(k=5, metric="cosine"),
    SoftImputeImputer(max_rank=64, input_transform="centered"),
]

baseline_predictions = {}   # name → list[Prediction]

for baseline in BASELINES:
    print(f"Fitting {baseline.name}…", end=" ", flush=True)
    baseline.fit(kb)
    preds = baseline.impute(kb, test_pairs)
    baseline_predictions[baseline.name] = preds
    print("done")

## 5  Evaluate and report

In [ ]:
evaluator = Evaluator()
all_reports = {}

for name, preds in baseline_predictions.items():
    report = evaluator.evaluate(gold_items, preds)
    all_reports[name] = report

# Print accuracy by resource group for each baseline
groups = ["lrl", "mrl", "hrl", "overall"]
header = f"{'baseline':<15}" + "".join(f"  {g:>8}" for g in groups)
print(header)
print("-" * len(header))
for name, report in all_reports.items():
    row = f"{name:<15}"
    for g in groups:
        acc = report.get(g, {}).get("accuracy", float("nan"))
        row += f"  {acc:>8.3f}"
    print(row)

In [ ]:
BASELINES_OUT.parent.mkdir(parents=True, exist_ok=True)
with BASELINES_OUT.open("w", encoding="utf-8") as f:
    json.dump(all_reports, f, indent=2, ensure_ascii=False)

print(f"Report saved → {BASELINES_OUT}")